Setting Up the Directory

In [ ]:
import os

# Root directory for cloned repositories and the conda environment
os.environ['ROOT_DIR'] = '~/scratch'

# Path to AlphaFold data (pre-downloaded on PACE ICE)
os.environ['DATA_DIR'] = '/storage/ice1/shared/d-pace_community/alphafold/alphafold_2.3.2_data'

# Directory where the conda environment will be stored
os.environ['CONDA_INSTALL_DIR'] = '~/scratch'

Fix paths to be expanded from user to exact paths

In [ ]:
# Resolve ~ to the absolute home directory path for use in bash cells
os.environ['ROOT_DIR'] = os.path.expanduser(os.environ['ROOT_DIR'])
os.environ['DATA_DIR'] = os.path.expanduser(os.environ['DATA_DIR'])
os.environ['CONDA_INSTALL_DIR'] = os.path.expanduser(os.environ['CONDA_INSTALL_DIR'])

Clone the Vizfold-Foundation, and OpenFold repository.

In [ ]:
%%bash
# Clone VizFold-Foundation and OpenFold into the root directory
git clone https://github.com/AI2Science/vizfold-foundation.git $ROOT_DIR/vizfold-foundation
git clone https://github.com/aqlaboratory/openfold.git $ROOT_DIR/openfold

In [ ]:
%%bash
# Overwrite the upstream OpenFold environment.yml with a version that is
# compatible with PACE ICE (CUDA 12.4, system GCC, no conda-bundled pip packages).
cd $ROOT_DIR/openfold
cat > environment.yml << 'EOF'
name: openfold_env
channels:
  - pytorch
  - nvidia
  - conda-forge
  - bioconda
dependencies:
  - python=3.10
  - pytorch=2.5.1
  - pytorch-cuda=12.4
  # Build tools — use conda cross-compiler; system GCC is used at build time
  - gxx_linux-64
  - gcc_linux-64
  - libstdcxx-ng
  - sysroot_linux-64=2.17
  - make
  - ninja
  # CUDA development headers and libraries
  - nvidia::cuda-nvcc=12.4
  - nvidia::cuda-libraries-dev=12.4
  - nvidia::cuda-cudart-dev=12.4
  - nvidia::cuda-driver-dev=12.4
  # Core scientific stack
  - numpy
  - pandas
  - scipy
  - tqdm
  - pyyaml
  - requests
  - typing-extensions
  # ML / framework dependencies
  - wandb
  - pytorch-lightning
  # Structural biology tools
  - openmm
  - pdbfixer
  - biopython
  - modelcif==0.7
  # Sequence search databases and tools
  - bioconda::hmmer
  - bioconda::hhsuite
  - bioconda::kalign2
  # Utilities
  - awscli
  - ml-collections
  - aria2
  - git
  - pip
  - packaging
EOF
echo "environment.yml written successfully"

Create and activate the OpenFold conda environment

In [ ]:
%%bash
cd $ROOT_DIR/openfold
export PYTHONNOUSERSITE=1
export MAX_JOBS=4
source "$(conda info --base)/etc/profile.d/conda.sh"
export CONDA_ENVS_PATH=$CONDA_INSTALL_DIR/.conda/envs
export CONDA_PKGS_DIRS=$CONDA_INSTALL_DIR/.conda/pkgs

# Remove any previous (possibly partial) environment
echo "Removing old environment if it exists..."
conda env remove -n openfold_env -y || true

# Create the environment from the patched environment.yml
echo "Creating conda environment..."
conda env create -f environment.yml --force

# Install pip-only packages after the conda environment is ready.
conda activate openfold_env
echo "Installing pip dependencies..."
pip install deepspeed==0.14.5 dm-tree==0.1.6 git+https://github.com/NVIDIA/dllogger.git
pip install flash-attn --no-build-isolation --no-cache-dir

# ESMFold dependencies
echo "Installing ESMFold dependencies..."
pip install transformers==4.38.2 safetensors regex

echo "openfold_env created successfully"

Set up compiler and library paths

In [ ]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
export CONDA_ENVS_PATH=$CONDA_INSTALL_DIR/.conda/envs
export CONDA_PKGS_DIRS=$CONDA_INSTALL_DIR/.conda/pkgs
conda activate openfold_env
cd $ROOT_DIR/openfold

# Use the PACE ICE system GCC 12.3.0 instead of the conda-bundled compiler.
# The conda GCC toolchain causes linker issues when building CUDA extensions on RHEL 9.
export CC=/usr/local/pace-apps/spack/packages/linux-rhel9-x86_64_v3/gcc-11.3.1/gcc-12.3.0-ukkkutsxfl5kpnnaxflpkq2jtliwthfz/bin/gcc
export CXX=/usr/local/pace-apps/spack/packages/linux-rhel9-x86_64_v3/gcc-11.3.1/gcc-12.3.0-ukkkutsxfl5kpnnaxflpkq2jtliwthfz/bin/g++

export CFLAGS="-I${CONDA_PREFIX}/include"
export CXXFLAGS="-I${CONDA_PREFIX}/include"
export LDFLAGS="-L${CONDA_PREFIX}/lib"
export LIBRARY_PATH=${CONDA_PREFIX}/lib:${LIBRARY_PATH}
export LD_LIBRARY_PATH=${CONDA_PREFIX}/lib:${LD_LIBRARY_PATH}
export PYTHONNOUSERSITE=1
export MAX_JOBS=4

echo "Cleaning previous build artifacts..."
rm -rf build/ openfold.egg-info/ dist/

echo "Building OpenFold with conda GCC wrappers..."
pip install -e . --no-build-isolation

echo "Installing third-party dependencies..."
bash scripts/install_third_party_dependencies.sh

echo "OpenFold installation complete"

Set up Vizfold-Foundation

In [ ]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
export CONDA_ENVS_PATH=$CONDA_INSTALL_DIR/.conda/envs
export CONDA_PKGS_DIRS=$CONDA_INSTALL_DIR/.conda/pkgs
conda activate openfold_env

cd $ROOT_DIR/vizfold-foundation/openfold
mkdir -p resources

# Symlink to the shared AlphaFold parameters on PACE ICE
ln -sf $DATA_DIR/params resources/params

# Download stereo chemical properties file (required by OpenFold relaxation)
if [ ! -f "resources/stereo_chemical_props.txt" ]; then
    wget -q --no-check-certificate -P resources \
        https://git.scicore.unibas.ch/schwede/openstructure/-/raw/7102c63615b64735c4941278d92b554ec94415f8/modules/mol/alg/src/stereo_chemical_props.txt
fi

echo "VizFold-Foundation setup complete"

Install additional visualization tools

In [ ]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
export CONDA_ENVS_PATH=$CONDA_INSTALL_DIR/.conda/envs
export CONDA_PKGS_DIRS=$CONDA_INSTALL_DIR/.conda/pkgs
conda activate openfold_env

# Matplotlib for plotting attention maps
conda install -y conda-forge::matplotlib

# PyMOL for 3D molecular structure visualization
conda install -y -c conda-forge pymol-open-source

echo "Visualization tools installed successfully"